<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 6 - RL**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/PVAR_japan_endogenous_money

### **1) REQUIREMENTS SET-UP**

In [118]:
# Requirements.txt file installation
# !pip install -r requirements.txt

In [119]:
# Libraries Import
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

### **2) IMPORT & RESTRUCTURING**

In [120]:
# Transformed df Import
jp_trans_df = pd.read_csv("Data/Transformed/jp_trans_df.csv")

In [121]:
# Variables Remapping 
col_map = {
    "LogDiff-Monetary Aggregates - M1 (JPY)": "M1",
    "LogDiff-Monetary Aggregates - M2 (JPY)": "M2",
    "LogDiff-Monetary Aggregates - M3 (JPY)": "M3",
    "LogDiff-Total Treasury Reserves (- Gold)": "reserves",
    "LogDiff-BoJ’s Total Assets (100 Million Yen)": "boj_assets",
    "LogDiff-USD-JPY reer CPI-based (Index 2015=100)": "reer",
    "LogDiff-JPY-USD Spot Exchange Rate": "fx",
    "LogDiff-HICP (SA)": "inflation",
    "LogDiff-1615.T-Price": "nikkei",                            
    "LogDiff-Call Money/Interbank Immediate (%)": "call_rate_logdiff",
    "AR(1)detrend-Total Credit - General Government (%GDP)": "credit_gov",
    "AR(1)detrend-Total Credit - Households & NPISHs (%GDP)": "credit_hh",
    "AR(1)detrend-Total Credit - Private Non-Financial (%GDP)": "credit_pnf",
    "AR(1)detrend-10-Year Gov Bond Yields (%)": "10y_bond_yield",
    "AR(1)detrend-Call Money/Interbank Immediate (%)": "call_rate",
    "AR(1)detrend-Est. 1-year Neutral Interest Rate (%)": "neutral_rate_1y",
    "AR(1)detrend-Est. 10-year Neutral Interest Rate (%)": "neutral_rate_10y",
    "AR(1)detrend-Central Government Debt (% GDP)": "gov_debt",
    "AR(1)detrend-Domestic Private Debt Securities (% GDP)": "priv_debt_sec",
    "AR(1)detrend-Domestic Public Debt Securities (% GDP)": "pub_debt_sec",
    "AR(1)detrend-Loan Interest Rate (%)": "loan_rate",
    "AR(1)detrend-Deposit Interest Rate (%)": "deposit_rate",
    "AR(1)detrend-10-Year US T-Bills Yield (%)": "us_tbill",
    "AR(1)detrend-CBOE-VIX": "vix",
    "HPfilter-Real GDP (billions chained 2015 JPY)": "output_gap",
}
jp_core_df = jp_trans_df.copy().rename(columns=col_map)
jp_core_df.to_csv("Data/jp_core_df.csv", index=False)
jp_core_df

,Country,Time,M1,M2,M3,reserves,reer,fx,inflation,nikkei,...,neutral_rate_1y,neutral_rate_10y,gov_debt,priv_debt_sec,pub_debt_sec,loan_rate,deposit_rate,us_tbill,vix,output_gap
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.012713,0.004976,NaN,0.043927,...,0.002655,0.002863,NaN,NaN,NaN,0.264511,0.004569,0.041292,-4.176314,-0.001618
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.012815,0.004706,NaN,0.124617,...,NaN,NaN,NaN,NaN,NaN,NaN,0.006569,0.062536,0.032409,NaN
902,JP,2026-02,NaN,NaN,NaN,0.007443,0.000443,-0.009937,NaN,0.079644,...,NaN,NaN,NaN,NaN,NaN,NaN,0.073583,-0.095179,2.525659,NaN
903,JP,2026-03,NaN,NaN,NaN,NaN,NaN,0.022826,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.112260,6.349080,NaN


### **3) STATE VARIABLE CONSTRUCTION**

In [122]:
# State Variable Construction
df = jp_core_df.copy()

# /////////////
# Bank of Japan
# /////////////
# (!!!) The BoJ range of action incorporates both the continuous range of selection of the interest rate (policy intrument) and the adoption of a policy regime (ycc/qe)
df["qe_proxy"] = df["boj_assets"].fillna(0)

# ////////////////
# Commercial Banks
# ////////////////
# Net Interest Margin (banks" profit margin)
df["nim"] = df["loan_rate"] - df["deposit_rate"]
# Default proxy
# (!!!) Negative function of the credit_pnf growth rate, as a decrease of credit can be interpretated as an increase in the probability of firms default
# (!!!) It can"t go below zero
df["default_proxy"]  = (-df["credit_pnf"].fillna(0)).clip(lower=0)
# Capital Stress
# (!!!) lower nim increases capital stress, vix increases, and credit to pnf decreases 
df["capital_stress"] = (-df["nim"].fillna(0)
                        + df["vix"].fillna(0)
                        - df["credit_pnf"].fillna(0))

# //////////
# Households
# //////////
# Households" Debt Accumulation Proxy
df["hh_debt_proxy"] = df["credit_hh"].fillna(0).rolling(24, min_periods=6).sum()
# Real Cost of Borrowing for Households
df["real_rate"] = df["call_rate"] - df["inflation"]

In [123]:
# Set of Core Variables Construction
core_cols = [
    "inflation",
    "call_rate",
    "output_gap",
    "credit_pnf",
    "credit_hh",
    "gov_debt",
    "fx",
    "vix",
    "loan_rate",
    "deposit_rate",
    "nim",
    "boj_assets",
    "10y_bond_yield",
    "real_rate",
    "M3", 
    "qe_proxy",
    "default_proxy",
    "capital_stress",
    "hh_debt_proxy"
]
jp_state_df = df[["Country", "Time"] + core_cols].dropna().copy()
jp_state_df.to_csv("Data/jp_state_df.csv", index=False)
jp_state_df

,Country,Time,inflation,call_rate,output_gap,credit_pnf,credit_hh,gov_debt,fx,vix,...,deposit_rate,nim,boj_assets,10y_bond_yield,real_rate,M3,qe_proxy,default_proxy,capital_stress,hh_debt_proxy
569,JP,1998-05,1.032947,-0.010446,-0.007774,0.037320,-0.004204,-0.439234,0.023571,-0.765213,...,-0.094829,0.086047,-0.036587,-0.205574,-1.043393,0.004179,-0.036587,0.000000,-0.888580,-0.129739
570,JP,1998-06,-0.427880,0.013796,-0.006858,0.037320,-0.004204,-0.439234,0.039496,1.002092,...,0.013456,0.005699,-0.023495,-0.110000,0.441675,0.000768,-0.023495,0.000000,0.959074,-0.129739
571,JP,1998-07,-3.399241,-0.018700,-0.004739,1.337320,0.095796,-0.439234,0.003251,-1.399135,...,0.100502,-0.044476,-0.040035,0.147226,3.380541,0.004922,-0.040035,0.000000,-2.691980,-0.129739
572,JP,1998-08,-4.032639,0.018703,-0.004053,0.041528,-0.003848,-0.439234,0.027273,11.727565,...,0.082153,-0.087064,0.052921,-0.175812,4.051342,0.004464,0.052921,0.000000,11.773101,-0.129739
573,JP,1998-09,5.911240,-0.100781,-0.003498,0.041528,-0.003848,-0.439234,-0.073105,8.463197,...,-0.206328,0.195482,0.025754,-0.395030,-6.012020,0.003470,0.025754,0.000000,8.226187,-0.129739
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849,JP,2021-09,5.691701,0.013023,-0.000929,-0.035191,-0.016309,-0.405006,0.002803,2.046939,...,0.001344,0.053322,-0.003656,0.038998,-5.678677,0.002400,-0.003656,0.035191,2.028808,4.009240
850,JP,2021-10,-7.746593,-0.003877,0.009121,1.064809,-0.016309,-0.405006,0.026520,-1.899905,...,0.004282,-0.060265,0.001560,0.024301,7.742716,0.003444,0.001560,0.000000,-2.904450,3.525262
851,JP,2021-11,2.639222,-0.010918,0.007280,-0.031630,-0.016309,-0.405006,0.007429,0.384726,...,0.001091,-0.052362,0.004170,-0.050498,-2.650141,0.002083,0.004170,0.031630,0.468718,3.539503
852,JP,2021-12,0.904571,0.013982,0.005531,-0.031630,-0.016309,-0.405006,-0.001160,2.705635,...,0.004227,0.048406,-0.006133,0.014200,-0.890589,0.002447,-0.006133,0.031630,2.688858,3.553745


### **4) TRANSITION DYNAMICS**

In [124]:
# Transition Dynamics 

df = jp_state_df.copy() 

# Lead Extraction 
# (!!!) We define the dependent side of the regression as the lead t+1 of each variable to assess how the state evolves following the impact of the action bundle executed by the agents 
# Output Gap
df["output_gap_t+1"] = df["output_gap"].shift(-1)
# Inflation
df["inflation_t+1"] = df["inflation"].shift(-1)
# Default Proxy
df["default_proxy_t+1"] = df["default_proxy"].shift(-1)
# Capital Stress
df["capital_stress_t+1"] = df["capital_stress"].shift(-1)
# Household Debt Proxy
df["hh_debt_proxy_t+1"] = df["hh_debt_proxy"].shift(-1)
# Negative Component Output Gap 
# (!!!) It takes the maximum between 0 and recession periods that since they have a minus in front (-df["output_gap_t+1"]) become negative and then are teaken into account, otherwise during normal periods of positive growth, the default rate is indifferent so 0
df["negative_output_gap_t+1"] = np.maximum(0, -df["output_gap_t+1"].fillna(0))
# Credit Efficiency
# (!!!) The minimum between what banks supply and what households demand
# (!!!) Even if the BoJ floods banks with reserves, if households are deleveraging (credit_hh < 0), effective credit is negative regardless
df["credit_eff"] = np.minimum(df["credit_pnf"].fillna(0),
                              df["credit_hh"].fillna(0))

# ////////////////////////////////////
# Equation 1 — Output Gap Transmission
# ////////////////////////////////////
X1 = sm.add_constant(df[["output_gap","credit_eff","capital_stress"]].rename(
     columns={"output_gap":"output_gap_t","credit_eff":"credit_eff_t","capital_stress":"capital_stress_t"}))
m1 = sm.OLS(df["output_gap_t+1"], X1, missing="drop").fit()

# ///////////////////////////////////
# Equation 2 — Inflation Transmission
# ///////////////////////////////////
# (!!!) From equation 1 the outgap change transmits to inflation
X2 = sm.add_constant(df[["inflation","output_gap_t+1"]].rename(
     columns={"inflation":"inflation_t"}))
m2 = sm.OLS(df["inflation_t+1"], X2, missing="drop").fit()

# ///////////////////////////////////////
# Equation 3 — Default Proxy Transmission
# ///////////////////////////////////////
# (!!!) We transform output gap to negative, as the contraction of output tends to lead to higher distress for firms corresponding to expected higher default rates
# (!!!) Commercial Banks Default Index (impossible to find a better proxy)
# (!!!) More credit, more exposure
X3 = sm.add_constant(df[["default_proxy","negative_output_gap_t+1","credit_pnf"]].rename(
     columns={"default_proxy":"default_proxy_t", "credit_pnf":"credit_pnf_t"}))
m3 = sm.OLS(df["default_proxy_t+1"], X3, missing="drop").fit()

# ////////////////////////////////////////  
# Equation 4 — Capital Stress Transmission
# ////////////////////////////////////////
# (!!!) Rising defaults directly tighten capital, losses eat into equity
# (!!!) NIM compression (from rate cuts) squeezes the bank's ability to rebuild capital organically through retained earnings. This creates the NIM-capital-lending doom loop that Japan experienced: cuts → lower NIM → weaker banks → less lending
X4 = sm.add_constant(df[["capital_stress","default_proxy_t+1","nim"]].rename(
     columns={"capital_stress":"capital_stress_t", "nim":"nim_t"}))
m4 = sm.OLS(df["capital_stress_t+1"], X4, missing="drop").fit()

# ////////////////////////////////////////  
# Equation 5 — Household Debt Transmission
# ////////////////////////////////////////
X5 = sm.add_constant(df[["hh_debt_proxy","credit_hh","real_rate"]].rename(
     columns={"hh_debt_proxy":"hh_debt_proxy_t", "credit_hh":"credit_hh_t", "real_rate":"real_rate_t"}))
m5 = sm.OLS(df["hh_debt_proxy_t+1"], X5, missing="drop").fit()

In [125]:
# Equations' Model Params Loop 
for target, model in [("Output Gap t+1", m1), 
                      ("Inflation t+1", m2), 
                      ("Default Proxy t+1", m3), 
                      ("Capital Stress t+1", m4), 
                      ("Household Debt Proxy t+1", m5)]:
    print(f"\n////////////////////// {target} (R²={model.rsquared:.4f}) //////////////////////")
# (!!!) .tables[1] is great to summarize the core statistics (params, p-values) of OLS results
    print(model.summary().tables[1])


////////////////////// Output Gap t+1 (R²=0.5613) //////////////////////
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -0.0001      0.000     -0.260      0.795      -0.001       0.001
output_gap_t         0.7518      0.041     18.264      0.000       0.671       0.833
credit_eff_t        -0.0006      0.001     -1.217      0.225      -0.002       0.000
capital_stress_t    -0.0003    9.3e-05     -3.670      0.000      -0.001      -0.000

////////////////////// Inflation t+1 (R²=0.0418) //////////////////////
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0620      0.164      0.377      0.706      -0.262       0.386
inflation_t        0.1633      0.059      2.764      0.006       0.047       0.280
output_gap_

In [126]:
# Model Transition Params Storage
# (!!!) Params saving for agents training 
transition_params_ols = {
# From Equation 1
    "alpha": m1.params.get("output_gap_t").round(4),
    "beta": m1.params.get("credit_eff_t").round(4),
    "gamma": m1.params.get("capital_stress_t").round(4),
# From Equation 2
    "rho": m2.params.get("inflation_t").round(4),
    "lam": m2.params.get("output_gap_t+1").round(4),
# From Equation 3
    "delta": m3.params.get("default_proxy_t").round(4),
    "phi": m3.params.get("negative_output_gap_t+1").round(4),
    "psi": m3.params.get("credit_pnf_t").round(4),
# From Equation 4
    "kappa": m4.params.get("capital_stress_t").round(4),
    "theta": m4.params.get("default_proxy_t+1").round(4),
    "mu": m4.params.get("nim_t").round(4),
# From Equation 5
    "delta_D": m5.params.get("hh_debt_proxy_t").round(4),
    "zeta": m5.params.get("credit_hh_t").round(4),
    "xi": m5.params.get("real_rate_t").round(4)
}
print(transition_params_ols)

{'alpha': 0.7518, 'beta': -0.0006, 'gamma': -0.0003, 'rho': 0.1633, 'lam': 26.9646, 'delta': -0.2198, 'phi': -12.3928, 'psi': -0.0544, 'kappa': 0.0246, 'theta': 1.2051, 'mu': -2.2835, 'delta_D': 0.9892, 'zeta': 0.0232, 'xi': -0.0064}


### **5) INVERSE REINFORCEMENT LEARNING AGENTS' REWARD FUNCTIONS (IRL)**

In [127]:
# Inverse Reinforcement Learning Agents' Reward Functions (IRL)
# (!!!) Infers each agent's implicit reward function from their observed historical actions, using linear Maximum-Entropy IRL
# (!!!) All features are standardised so coefficients are comparable across variables with different units
# (!!!) In normal RL we say: "here is the reward, go find the best action"
# (!!!) Inverse RL flips it: "here are the actions the agent actually took over 25 years, what reward function must they have been maximising?"

df = jp_state_df.copy()
df = df.set_index(pd.to_datetime(df["Time"])).drop(columns="Time")

# BoJ explicit inflation target (annual 2%)
# https://www.boj.or.jp/en/mopo/outline/target.htm
pi_star = 2

# Derived metrics
# BoJ Reward Function, inflation component (the second one is outpuit stabilization, both of them weighted)
df["inflation_gap"] = df["inflation"] - pi_star
# BoJ Action 1: call rate variation 
df["boj_action1_rate"] = df["call_rate"].diff()
# BoJ Action 2: Quantitative Easing/YCC 
df["boj_action2_qe"] = df["qe_proxy"]
# Commercial Bank Action 1: Credit supply 
df["bank_action1"] = df["credit_pnf"]
# Households Action 1: Credit demand (temporal budget constraint)
df["hh_action1"] = df["credit_hh"]

In [128]:
# ///////////////////
# Bank of Japan - IRL
# ///////////////////
# (!!!) You observe that the BoJ cut rates in 1999. Why? Because inflation was too low? Because output was falling? Because banks were stressed? 
boj_features = ["inflation_gap","output_gap", "capital_stress", "gov_debt", "fx", "vix"]
# BoJ set of actions (dependent)
boj_df = df[["boj_action1_rate", "boj_action2_qe"] + boj_features].dropna()

# Rescaling/Normalization 
# (!!!) We look into feature columns, learn their average and spread, and then rescale them
boj_scaler = StandardScaler()
boj_X = pd.DataFrame(boj_scaler.fit_transform(boj_df[boj_features]),
                     columns=boj_features, index=boj_df.index)

# Inverse Inference
# (!!!) How the BoJ acted in the past with respect to its range of actions while observing certain levels of features 
m_boj_rate = sm.OLS(boj_df["boj_action1_rate"], sm.add_constant(boj_X)).fit()
m_boj_qe = sm.OLS(boj_df["boj_action2_qe"], sm.add_constant(boj_X)).fit()

In [129]:
# /////////////////////
# Commercial Bank - IRL
# /////////////////////
bank_features = ["nim", "capital_stress", "default_proxy", "call_rate", "credit_hh"]
bank_df = df[["bank_action1"] + bank_features].dropna()

bank_scaler = StandardScaler()
bank_X = pd.DataFrame(bank_scaler.fit_transform(bank_df[bank_features]),
                      columns=bank_features, index=bank_df.index)

m_bank = sm.OLS(bank_df["bank_action1"], sm.add_constant(bank_X)).fit()

In [130]:
# ////////////////
# Households - IRL
# ////////////////
hh_features = ["real_rate", "output_gap", "inflation", "deposit_rate", "hh_debt_proxy"]
hh_df = df[["hh_action1"] + hh_features].dropna()

hh_scaler = StandardScaler()
hh_X = pd.DataFrame(hh_scaler.fit_transform(hh_df[hh_features]),
                    columns=hh_features, index=hh_df.index)

m_hh = sm.OLS(hh_df["hh_action1"], sm.add_constant(hh_X)).fit()

In [131]:
# IRL' Model Params Loop 
for target, model in [("BoJ Rate", m_boj_rate), 
                      ("BoJ QE", m_boj_qe), 
                      ("Bank", m_bank), 
                      ("Household", m_hh)]:
    print(f"\n////////////////////// {target} (R²={model.rsquared:.4f}) //////////////////////")
# (!!!) .tables[1] is great to summarize the core statistics (params, p-values) of OLS results
    print(model.summary().tables[1])


////////////////////// BoJ Rate (R²=0.0309) //////////////////////
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const           6.175e-05      0.002      0.034      0.973      -0.003       0.004
inflation_gap      0.0008      0.002      0.435      0.664      -0.003       0.004
output_gap        -0.0006      0.002     -0.300      0.764      -0.004       0.003
capital_stress    -0.0109      0.008     -1.412      0.159      -0.026       0.004
gov_debt          -0.0008      0.002     -0.425      0.671      -0.004       0.003
fx                -0.0016      0.002     -0.850      0.396      -0.005       0.002
vix                0.0062      0.008      0.819      0.413      -0.009       0.021

////////////////////// BoJ QE (R²=0.0534) //////////////////////
                     coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------

In [132]:
# IRL Inferred Omega 
# (!!!) how much does this agent react to this variable?

# Bank of Japan 
omega_boj_rate = [m_boj_rate.params.get("inflation_gap").round(4),
                  m_boj_rate.params.get("output_gap").round(4),
                  m_boj_rate.params.get("capital_stress").round(4),
                  m_boj_rate.params.get("gov_debt").round(4)]
omega_boj_qe = [m_boj_qe.params.get("inflation_gap").round(4),
                m_boj_qe.params.get("output_gap").round(4),
                m_boj_qe.params.get("capital_stress").round(4),
                m_boj_qe.params.get("gov_debt").round(4)]

# Commercial Bank
omega_bank = [m_bank.params.get("nim").round(4),
              m_bank.params.get("capital_stress").round(4),
              m_bank.params.get("default_proxy").round(4),
              m_bank.params.get("credit_hh").round(4)]

# Households
omega_hh = [m_hh.params.get("real_rate").round(4),
            m_hh.params.get("output_gap").round(4),
            m_hh.params.get("inflation").round(4),
            m_hh.params.get("deposit_rate").round(4),
            m_hh.params.get("hh_debt_proxy").round(4)]

print(f"\n////////////////////// Bank of Japan - Omega Rate //////////////////////")
print(omega_boj_rate)
print(f"\n////////////////////// Bank of Japan - Omega QE //////////////////////")
print(omega_boj_qe)
print(f"\n////////////////////// Bank - Omega //////////////////////")
print(omega_bank)
print(f"\n////////////////////// Household - Omega //////////////////////")
print(omega_hh)


////////////////////// Bank of Japan - Omega Rate //////////////////////
[0.0008, -0.0006, -0.0109, -0.0008]

////////////////////// Bank of Japan - Omega QE //////////////////////
[0.0013, 0.0008, -0.0323, -0.0051]

////////////////////// Bank - Omega //////////////////////
[-0.0222, -0.1248, -0.6705, 0.4329]

////////////////////// Household - Omega //////////////////////
[-0.6837, -0.0943, -0.6703, 0.0087, 0.0723]


### **6) REGIME-SHIFTING IRL AGENTS' REWARD FUNCTIONS**

In [133]:
# Regime-shifting IRL
# (!!!) We rerun the same IRL models but on restricted periods to see spot non linearities 
regimes_irl = {"Pre-2000": ("1998-05", "1999-12"),
               "Lost Decade": ("2000-01", "2012-12"),
               "Abenomics": ("2013-01", "2016-01"),
               "YCC": ("2016-02", "2022-01")}

# Regimes-Shifting Loop (no-agent specific)
for regime, (regime_start, regime_end) in regimes_irl.items():

# Dataset slicing
# (!!!) Conversion to Timestamp
    t_start = pd.Timestamp(regime_start)
    t_end   = pd.Timestamp(regime_end) + pd.offsets.MonthEnd(0)
# Boolean Mask
    boj_slice  = boj_df[(boj_df.index  >= t_start) & (boj_df.index  <= t_end)]
    hh_slice   = hh_df[(hh_df.index   >= t_start) & (hh_df.index   <= t_end)]
    bank_slice = bank_df[(bank_df.index >= t_start) & (bank_df.index <= t_end)]

# ///////////////////////////////////
# Bank of Japan - Regime-Shifting IRL
# ///////////////////////////////////
    boj_X = pd.DataFrame(
        boj_scaler.transform(boj_slice[boj_features]),
        columns = boj_features,
        index   = boj_slice.index
    )
    m_boj_rate = sm.OLS(boj_slice["boj_action1_rate"], sm.add_constant(boj_X)).fit()
    m_boj_qe = sm.OLS(boj_slice["boj_action2_qe"], sm.add_constant(boj_X)).fit()

# /////////////////////////////////////
# Commercial Bank - Regime-Shifting IRL
# /////////////////////////////////////
    bank_X = pd.DataFrame(
        bank_scaler.transform(bank_slice[bank_features]),
        columns = bank_features,
        index   = bank_slice.index
    )
    m_bank = sm.OLS(bank_slice["bank_action1"], sm.add_constant(bank_X)).fit()

# ////////////////////////////////
# Households - Regime-Shifting IRL
# ////////////////////////////////
    hh_X = pd.DataFrame(
        hh_scaler.transform(hh_slice[hh_features]),
        columns = hh_features,
        index   = hh_slice.index
    )
    m_hh = sm.OLS(hh_slice["hh_action1"], sm.add_constant(hh_X)).fit()


# IRL Inferred Omega 
# (!!!) how much does this agent react to this variable?
# Bank of Japan 
    omega_boj_rate = [m_boj_rate.params.get("inflation_gap").round(4),
                      m_boj_rate.params.get("output_gap").round(4),
                      m_boj_rate.params.get("capital_stress").round(4),
                      m_boj_rate.params.get("gov_debt").round(4)]
    omega_boj_qe = [m_boj_qe.params.get("inflation_gap").round(4),
                    m_boj_qe.params.get("output_gap").round(4),
                    m_boj_qe.params.get("capital_stress").round(4),
                    m_boj_qe.params.get("gov_debt").round(4)]
# Commercial Bank
    omega_bank = [m_bank.params.get("nim").round(4),
                  m_bank.params.get("capital_stress").round(4),
                  m_bank.params.get("default_proxy").round(4),
                  m_bank.params.get("credit_hh").round(4)]
# Households
    omega_hh = [m_hh.params.get("real_rate").round(4),
                m_hh.params.get("output_gap").round(4),
                m_hh.params.get("inflation").round(4),
                m_hh.params.get("deposit_rate").round(4),
                m_hh.params.get("hh_debt_proxy").round(4)]

    # ── Print row ─────────────────────────────────────────────────────────────
    print(f"\n====================== {regime} ({regime_start}-{regime_end}) ======================")
    print(f"\n////////////////////// Bank of Japan - Omega Rate //////////////////////")
    print(omega_boj_rate)
    print(f"\n////////////////////// Bank of Japan - Omega QE //////////////////////")
    print(omega_boj_qe)
    print(f"\n////////////////////// Bank - Omega //////////////////////")
    print(omega_bank)
    print(f"\n////////////////////// Household - Omega //////////////////////")
    print(omega_hh)


====================== Pre-2000 (1998-05-1999-12) ======================

////////////////////// Bank of Japan - Omega Rate //////////////////////
[-0.0131, 0.0592, 0.0819, -0.0255]

////////////////////// Bank of Japan - Omega QE //////////////////////
[0.0051, -0.0555, -0.0023, -0.0381]

////////////////////// Bank - Omega //////////////////////
[-0.1178, -0.0283, -0.9024, 0.5676]

////////////////////// Household - Omega //////////////////////
[-5.245, 0.1757, -5.2532, 0.0255, 0.2727]

====================== Lost Decade (2000-01-2012-12) ======================

////////////////////// Bank of Japan - Omega Rate //////////////////////
[0.0032, -0.003, -0.0108, 0.0004]

////////////////////// Bank of Japan - Omega QE //////////////////////
[-0.0027, 0.004, -0.0611, -0.0025]

////////////////////// Bank - Omega //////////////////////
[0.0355, 0.1052, -0.8193, 0.0838]

////////////////////// Household - Omega //////////////////////
[-1.6369, -0.1118, -1.6157, 0.0026, 0.0524]

==========

### **7) MULTI-AGENT ENVIRONMENT**

In [ ]:
class JapanMacroEnv:
    """
    State (9-dim): [g, pi, i, u, d, a_bank, nim, c_hh, D_hh]
    Turn order:  1. Household → 2. Bank → 3. BoJ
    """
    def __init__(self, p):
        self.p = p; self.reset()

    def reset(self, s0=None):
        self.state = (np.zeros(9,dtype=np.float32)
                      if s0 is None else np.array(s0,dtype=np.float32))
        self.L=1.0; self.t=0
        return self.state.copy()

    def hh_step(self, c_hh):
        g,pi,i,u,d,a_bank,nim,_,D_hh = self.state; p=self.p
        real_r = i-pi
        D_hh_new = np.clip(p['delta_D']*D_hh + p['zeta']*c_hh - p['xi']*real_r,-50,50)
        pi_gap = pi-p.get('pi_star',0.02/12)
        ow = p['omega_hh']
        R_hh = (ow[0]*(-real_r)*abs(c_hh) + ow[1]*g*abs(c_hh)
                + ow[2]*(-pi_gap**2) + ow[3]*(-abs(D_hh_new)))
        if D_hh_new > p.get('D_max',5.): R_hh -= (D_hh_new-p.get('D_max',5.))*2.
        self.state[7]=float(c_hh); self.state[8]=float(D_hh_new)
        return self.state.copy(), float(R_hh)

    def bank_step(self, a_bank):
        g,pi,i,u,d,_,nim,c_hh,D_hh = self.state; p=self.p
        credit_eff = float(np.minimum(a_bank,c_hh))
        self.L = max(0.5, self.L*(1+d+credit_eff))
        g_new  = np.clip(p['alpha']*g+p['beta']*credit_eff-p['gamma_g']*(-u)
                         +np.random.normal(0,.001),-0.1,0.1)
        pi_new = np.clip(p['rho']*pi+p['lam']*g_new
                         +np.random.normal(0,.0005),-0.15,0.15)
        d_new  = np.clip(p['delta']*d+p['phi']*max(0.,-g_new)+p['psi']*a_bank,0.,2.)
        losses   = d_new*self.L*0.01
        retained = max(0.,nim*self.L*0.01)
        u_new  = np.clip(p['kappa']*u+p['theta']*d_new+p['mu']*nim+retained-losses,-10,10)
        cap_pen = max(0.,p.get('u_min',-2.)-u_new)*5.
        ow = p['omega_bank']
        R_bank = (ow[0]*nim*abs(a_bank)+ow[1]*(-u_new)*0.1
                  +ow[2]*(-d_new)*abs(a_bank)+ow[3]*c_hh*abs(a_bank)-cap_pen)
        self.state=np.array([g_new,pi_new,i,u_new,d_new,a_bank,nim,c_hh,D_hh],dtype=np.float32)
        self.t+=1
        return self.state.copy(), float(R_bank)

    def boj_step(self, di_action, qe_action):
        g,pi,i,u,d,a_bank,nim,c_hh,D_hh = self.state; p=self.p
        i_new   = np.clip(i+di_action,-0.02,0.1)
        nim_new = np.clip(nim-di_action*0.3+qe_action*0.05,-0.1,0.1)
        pi_gap  = pi-p.get('pi_star',0.02/12)
        ow = p['omega_boj']
        R_boj = ow[0]*(-pi_gap**2)+ow[1]*g+ow[2]*(-abs(u))
        self.state[2]=float(i_new); self.state[6]=float(nim_new)
        return self.state.copy(), float(R_boj)

    def get_hh_obs(self):
        g,pi,i,u,d,a_bank,nim,c_hh,D_hh = self.state
        return np.array([pi-self.p.get('pi_star',0.02/12),i-pi,g,nim,D_hh,a_bank],
                        dtype=np.float32)
    def get_bank_obs(self): return self.state.copy()
    def get_boj_obs(self):
        g,pi,i,u,d,a_bank,nim,c_hh,D_hh = self.state
        return np.array([pi-self.p.get('pi_star',0.02/12),g,u,d,nim,c_hh,D_hh],
                        dtype=np.float32)